In [22]:
import pandas as pd

etapa 1
carregando os dados brutos

In [23]:
df_transacoes = pd.read_csv('transacoes_2026.csv')
df_clientes = pd.read_csv('clientes_cadastro.csv')

etapa 2
normalização de nulos e formatação dos nomes, para evitar erros de digitação e garantir que a junção funcione sem problemas

In [24]:
df_transacoes.columns = df_transacoes.columns.str.lower().str.strip()
df_clientes.columns = df_clientes.columns.str.lower().str.strip()

df_transacoes['valor_desconto'] = df_transacoes['valor_desconto'].fillna(0.0)
df_clientes = df_clientes.dropna(subset = ['id_cliente'])

etapa 3
conversão de tipos de dados, para garantir que datas nao funcionem como textos e valores numericos funcionem como ponto flutuante

In [25]:
 df_transacoes['data_transacao'] = pd.to_datetime(df_transacoes['data_transacao'])
 df_transacoes['valor_bruto'] = df_transacoes['valor_bruto'].astype(float)

etapa 4
junção relacional, nossa tabela tem o historico de vendas, mas nao tem o nome do cliente nem a cidade onde ele mora 
vamos unir as duas tabelas usando a coluna id_cliente como chave
usando um LEFT JOIN (how = left): para garantir que todas as transacoes serão mantidas

In [26]:
df_consolidado = pd.merge(df_transacoes, df_clientes, on = 'id_cliente', how = 'left')
df_consolidado


,id_transacao,id_cliente,data_transacao,valor_bruto,valor_desconto,status,nome_cliente,cidade
0,101,1.0,2026-03-01,120.0,10.0,Aprovada,Ana Silva,São Paulo
1,102,2.0,2026-03-02,45.0,0.0,Aprovada,Bruno Costa,Rio de Janeiro
2,103,1.0,2026-03-02,300.5,25.0,Aprovada,Ana Silva,São Paulo
3,104,3.0,2026-03-03,80.0,0.0,Recusada,Carla Souza,Belo Horizonte
4,105,NaN,2026-03-04,150.0,0.0,Aprovada,NaN,NaN
5,106,4.0,2026-03-05,200.0,15.0,Aprovada,Daniel Lima,Curitiba


etapa 5
calculando e criando a nova coluna valor liquido que igual o nome contem o quanto a loja realmente recebeu

In [28]:
df_consolidado['valor_liquido'] = df_consolidado['valor_bruto'] - df_consolidado['valor_desconto']

etapa 7
Filtragem final  selecionamos apenas transações de valor significativo 

In [29]:
data_final = df_consolidado[(df_consolidado['valor_liquido'] > 50.00) & (df_consolidado['status'].str.lower() == 'aprovada')]
data_final

,id_transacao,id_cliente,data_transacao,valor_bruto,valor_desconto,status,nome_cliente,cidade,valor_liquido
0,101,1.0,2026-03-01,120.0,10.0,Aprovada,Ana Silva,São Paulo,110.0
2,103,1.0,2026-03-02,300.5,25.0,Aprovada,Ana Silva,São Paulo,275.5
4,105,NaN,2026-03-04,150.0,0.0,Aprovada,NaN,NaN,150.0
5,106,4.0,2026-03-05,200.0,15.0,Aprovada,Daniel Lima,Curitiba,185.0


etapa 8
finalizacao e exportação


In [30]:
data_final.to_csv('base_consolidada_limpa.csv', index = False, encoding = 'utf-8-sig')